In [ ]:
from tqdm.notebook import tqdm
from prepare import prepare_project
from process import process_project
from score import score_project

datasets = {
    "visa": [
        "fryum"
    ],
    # "mvtec_ad":[""]
}
for source in datasets.keys():
    for project in tqdm(datasets[source]):
        prepare_project(source,project)
        process_project(project,force_from="segment")
        score_project(project)

  0%|          | 0/1 [00:00<?, ?it/s]

VisA dataset already exists at C:\Users\Matti\PycharmProjects\makroInspect\datasets\visa

Initializing project: fryum

Setting up project folder...
  ✓ config.yaml already exists, not overwriting
  ✓ Created runs.csv

Setting up data folder...

Reading VisA metadata...
  Found 600 images

Copying images...



Processing: 100%|██████████| 600/600 [00:00<00:00, 793.91it/s] 



✓ Project 'fryum' initialized

  Project folder (permanent):
    C:\Users\Matti\PycharmProjects\makroInspect\projects\fryum

  Artifacts folder (regenerable):
    C:\Users\Matti\PycharmProjects\makroInspect\artifacts\fryum

  Dataset stats:
    Train (normal):  450
    Test (normal):   50
    Test (anomaly):  100
    GT masks:        100
    Defect types:    burnt, corner or edge breakage, corner or edge breakage,small scratches, different colour spot, different colour spot,similar colour spot, fryum stuck together, middle breakage, middle breakage,similar colour spot, middle breakage,small scratches, similar colour spot, similar colour spot,other, similar colour spot,small scratches, small scratches
Processing project: fryum
Config: C:\Users\Matti\PycharmProjects\makroInspect\projects\fryum\config.yaml

Forcing rerun from 'segment': ['segment', 'crop', 'embed', 'bank', 'heatmap', 'refine']

Running: segment
  Prompts: ['foreground object']
  Multi-instance: True
  Device: cuda
  Foun

Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)


  'foreground object':   7%|▋         | 40/600 [00:06<01:44,  5.33img/s]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
from pathlib import Path

project = category
artifacts_root = Path(f"artifacts/{project}")
data_root = artifacts_root / "data"
heatmaps_root = artifacts_root / "heatmaps"

# Gather anomalous test images
anomaly_images = []
for label_dir in (data_root / "test").iterdir():
    if not label_dir.is_dir() or label_dir.name == "good":
        continue
    label = label_dir.name
    for img_path in sorted(label_dir.glob("*")):
        if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            stem = img_path.stem

            refined_path = heatmaps_root / "refined" / "test" / label / f"{stem}.npy"
            reverted_path = heatmaps_root / "reverted" / "test" / label / f"{stem}.npy"
            heatmap_path = refined_path if refined_path.exists() else reverted_path

            anomaly_images.append({
                "stem": stem,
                "label": label,
                "image": img_path,
                "gt": data_root / "ground_truth" / label / f"{stem}.png",
                "heatmap": heatmap_path,
                "is_refined": refined_path.exists(),
            })

n_refined = sum(1 for x in anomaly_images if x["is_refined"])
print(
    f"Found {len(anomaly_images)} anomalous test images ({n_refined} refined, {len(anomaly_images) - n_refined} reverted)")
print()

for i, entry in enumerate(anomaly_images):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    img = cv2.imread(str(entry["image"]))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    gt = cv2.imread(str(entry["gt"]), cv2.IMREAD_GRAYSCALE) if entry["gt"].exists() else None
    heatmap = np.load(entry["heatmap"]) if entry["heatmap"].exists() else None

    # Original
    axes[0].imshow(img_rgb)
    axes[0].set_title(f"{entry['label']}/{entry['stem']}")
    axes[0].axis("off")

    # Ground Truth
    if gt is not None:
        axes[1].imshow(gt, cmap="gray")
    else:
        axes[1].text(0.5, 0.5, "No GT", ha="center", va="center")
    axes[1].set_title("Ground Truth")
    axes[1].axis("off")

    # Overlay (heatmap on image)
    if heatmap is not None:
        heatmap_resized = cv2.resize(heatmap, (img_rgb.shape[1], img_rgb.shape[0]))
        denom = (heatmap_resized.max() - heatmap_resized.min()) + 1e-8
        heatmap_norm = (heatmap_resized - heatmap_resized.min()) / denom
        heatmap_color = (plt.cm.magma(heatmap_norm)[:, :, :3] * 255).astype(np.uint8)
        blended = (0.4 * img_rgb + 0.6 * heatmap_color).astype(np.uint8)
        axes[2].imshow(blended)
        source = "refined" if entry["is_refined"] else "reverted"
        axes[2].set_title(f"Overlay [{source}] (max={heatmap.max():.2f})")
    else:
        axes[2].imshow(img_rgb)
        axes[2].set_title("Overlay (no heatmap found)")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

    if i < len(anomaly_images) - 1:
        input(f"[{i + 1}/{len(anomaly_images)}] Press Enter for next...")
    else:
        print("Done!")